In [ ]:
!pip install streamlit -q
!pip install pyngrok -q
!ngrok authtoken <AUTHTOKEN>

In [ ]:
%%writefile main.py

import streamlit as st
import torch
from PIL import Image
from diffusers import LEditsPPPipelineStableDiffusion
from diffusers.utils import load_image
import io

# Set page config
st.set_page_config(page_title="LEDits++ Image Editor", layout="wide")

# Check GPU availability
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# Title and description
st.title("🎨 LEDits++ Image Editor")
st.markdown("""
Edit images using text prompts with the LEDits++ pipeline.
Upload an image, provide editing prompts, and adjust parameters to transform your image.
""")

# Display GPU status
if torch.cuda.is_available():
    st.success(f"🚀 GPU Available: {torch.cuda.get_device_name(0)} | Using: {DEVICE}")
else:
    st.info(f"💻 Running on CPU (GPU not available)")

# Initialize session state for pipeline
if 'pipeline' not in st.session_state:
    st.session_state.pipeline = None
    st.session_state.inverted = False
    st.session_state.original_image = None

# Sidebar for model loading
with st.sidebar:
    st.header("⚙️ Setup")

    model_choice = st.selectbox(
        "Model",
        ["stable-diffusion-v1-5/stable-diffusion-v1-5"],
        help="Choose the base model to use"
    )

    if st.button("Load Model"):
        with st.spinner("Loading model... This may take a while."):
            try:
                st.session_state.pipeline = LEditsPPPipelineStableDiffusion.from_pretrained(
                    model_choice,
                    torch_dtype=DTYPE
                )

                st.session_state.pipeline = st.session_state.pipeline.to(DEVICE)
                st.session_state.pipeline.enable_vae_tiling()

                st.success(f"✅ Model loaded successfully on {DEVICE.upper()}!")
            except Exception as e:
                st.error(f"Error loading model: {str(e)}")

    if st.session_state.pipeline:
        st.success(f"Model is ready on {DEVICE.upper()}!")
        if torch.cuda.is_available():
            # Display GPU memory info
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            gpu_allocated = torch.cuda.memory_allocated(0) / 1024**3
            gpu_reserved = torch.cuda.memory_reserved(0) / 1024**3
            st.info(f"GPU Memory: {gpu_allocated:.2f}GB / {gpu_memory:.2f}GB allocated")


# Main content
col1, col2 = st.columns(2)

with col1:
    st.header("📤 Input")

    # File uploader
    uploaded_file = st.file_uploader(
        "Upload an image",
        type=["jpg", "jpeg", "png"],
        help="Upload an image to edit"
    )

    if uploaded_file is not None:
        # Load and display original image
        image = Image.open(uploaded_file).convert("RGB")
        st.session_state.original_image = image
        st.image(image, caption="Original Image", use_container_width=True)

        # Inversion parameters
        st.subheader("Inversion Parameters")

        source_prompt = st.text_input(
            "Source Prompt (optional)",
            "",
            help="Prompt describing the input image. Leave empty to disable guidance during inversion."
        )

        source_guidance_scale = st.slider(
            "Source Guidance Scale",
            0.0, 10.0, 3.5, 0.5,
            help="Strength of guidance during inversion"
        )

        num_inversion_steps = st.slider(
            "Inversion Steps",
            10, 100, 50, 5,
            help="Number of inversion steps"
        )

        skip = st.slider(
            "Skip",
            0.0, 0.5, 0.15, 0.05,
            help="Portion of initial steps to skip. Lower values = stronger changes"
        )

        # Invert button
        if st.session_state.pipeline and st.button("🔄 Invert Image"):
            with st.spinner("Inverting image..."):
                try:
                    # Resize image to appropriate size
                    image_resized = image.resize((512, 512))

                    # Perform inversion
                    _ = st.session_state.pipeline.invert(
                        image=image_resized,
                        source_prompt=source_prompt,
                        source_guidance_scale=source_guidance_scale,
                        num_inversion_steps=num_inversion_steps,
                        skip=skip
                    )

                    st.session_state.inverted = True
                    st.success("✅ Image inverted successfully!")
                except Exception as e:
                    st.error(f"Error during inversion: {str(e)}")

with col2:
    st.header("🖼️ Output")

    if st.session_state.inverted:
        st.subheader("Editing Parameters")

        # Editing prompts
        editing_prompt = st.text_area(
            "Editing Prompt(s)",
            "cherry blossom",
            help="Enter one or more editing prompts (one per line for multiple)"
        )

        editing_prompts = [p.strip() for p in editing_prompt.split('\n') if p.strip()]

        # Edit guidance scale
        edit_guidance_scale = st.slider(
            "Edit Guidance Scale",
            1.0, 20.0, 10.0, 0.5,
            help="Strength of editing guidance"
        )

        # Edit threshold
        edit_threshold = st.slider(
            "Edit Threshold",
            0.0, 1.0, 0.75, 0.05,
            help="Masking threshold. Lower values = more editing"
        )

        # Edit warmup steps
        edit_warmup_steps = st.slider(
            "Edit Warmup Steps",
            0, 20, 0, 1,
            help="Number of initial steps without editing guidance"
        )

        # Reverse editing direction
        reverse_editing_direction = st.checkbox(
            "Reverse Editing Direction",
            False,
            help="Decrease instead of increase the editing prompt"
        )

        # Advanced options
        with st.expander("Advanced Options"):
            use_cross_attn_mask = st.checkbox("Use Cross-Attention Mask", False)
            use_intersect_mask = st.checkbox("Use Intersect Mask", True)
            guidance_rescale = st.slider("Guidance Rescale", 0.0, 1.0, 0.0, 0.1)

        # Generate button
        if st.button("✨ Generate Edited Image"):
            with st.spinner("Generating edited image..."):
                try:
                    output = st.session_state.pipeline(
                        editing_prompt=editing_prompts,
                        edit_guidance_scale=edit_guidance_scale,
                        edit_threshold=edit_threshold,
                        edit_warmup_steps=edit_warmup_steps,
                        reverse_editing_direction=reverse_editing_direction,
                        use_cross_attn_mask=use_cross_attn_mask,
                        use_intersect_mask=use_intersect_mask,
                        guidance_rescale=guidance_rescale,
                    )

                    # Display result
                    st.image(output.images[0], caption="Edited Image", use_container_width=True)

                    # Download button
                    buf = io.BytesIO()
                    output.images[0].save(buf, format="PNG")
                    st.download_button(
                        label="📥 Download Edited Image",
                        data=buf.getvalue(),
                        file_name="edited_image.png",
                        mime="image/png"
                    )

                except Exception as e:
                    st.error(f"Error during generation: {str(e)}")
    else:
        st.info("👈 Please upload an image and run inversion first")

# Footer
st.markdown("---")
st.markdown("""
### How to Use:
1. **Load Model**: Click "Load Model" in the sidebar
2. **Upload Image**: Upload an image you want to edit
3. **Invert**: Set inversion parameters and click "Invert Image"
4. **Edit**: Provide editing prompt(s) and parameters, then click "Generate Edited Image"

### Tips:
- Use descriptive editing prompts (e.g., "sunset", "snowy mountain", "cherry blossom")
- Lower edit threshold = more aggressive editing
- Higher edit guidance scale = stronger editing effect
- Use reverse editing direction to remove elements instead of adding them
""")

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
public_url

In [ ]:
!streamlit run main.py